# DIY Copilot
## Lesson: Model Context Protocol (MCP)

In the last lesson, our tool was a Python function **inside** our harness: we wrote its description, and our tool loop executed it. That works – but only for our harness. If you want to use the same knowledge base search in GitHub Copilot, Claude Code, Cursor or your colleague's assistant, each of these harnesses needs its own integration.

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) solves this. It is an open standard for connecting AI applications – like our harness – to tools and data:

```
 ┌──────────────────────────┐            ┌────────────────────────────────┐
 │ Harness (MCP host)       │            │ MCP server                     │
 │ e.g. our notebook,       │  MCP       │ e.g. company knowledge base    │
 │ Claude Code, VS Code     │ ◄────────► │                                │
 │   └─ MCP client          │ (JSON-RPC) │  tool: search_knowledge_base   │
 └──────────────────────────┘            └────────────────────────────────┘
```

* An **MCP server** offers tools (and optionally resources and prompts). It describes them itself.
* An **MCP client** – part of the harness, which the MCP specification calls the *host* – connects to servers, **discovers** their tools at runtime and calls them.
* Client and server talk **JSON-RPC**, either via stdin/stdout of a local process (`stdio`) or via HTTP for remote servers.

In this lesson we move **exactly the same function** from the last lesson – `search_knowledge_base(query)` – into an MCP server and connect our assistant to it.

During this lesson you will learn:
* how an MCP server is built and how it describes its tools
* what client and server actually exchange (under the hood)
* how to connect to a server, discover its tools and call them with the MCP SDK
* how to plug MCP tools into the tool loop from the last lesson
* how to write your own MCP server and use several servers at once
* what MCP servers offer besides tools: **resources** and **prompts**
* how other harnesses (Claude Code, VS Code, ...) can use the same server

>**Free tier:** This notebook makes about 8 LLM requests and a few embedding requests.

### Set up the environment

In [ ]:
# >>> SETUP <<<
# In Google Colab, this cell
#   1. clones the workshop repository,
#   2. installs its dependencies into the Python environment of this notebook (with uv),
#   3. makes the shared code (lib) importable.
# Locally (after `uv sync` in the repository), only step 3 is needed.
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/openknowledge/diy-copilot-bootcamp"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/diy-copilot")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, str(REPO_DIR / "setup" / "install_dependencies.py")], check=True)
else:
    # The repository root is the first parent directory containing the shared code
    REPO_DIR = next(p for p in Path.cwd().resolve().parents if (p / "lib").is_dir() and (p / "pyproject.toml").exists())

# The repository is not an installed package, so we add it to the module search path
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Workshop repository: {REPO_DIR}")

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

from litellm import completion
from mcp.types import LATEST_PROTOCOL_VERSION

from lib.constants import PROJECT_ROOT, TICKETS_PATH
from lib.environment import load_environment
from lib.mcp_client import (
    SERVER_LOG,
    McpServer,
    call_tool,
    get_prompt,
    list_prompts,
    list_resources,
    list_tools,
    print_prompts,
    print_resources,
    print_tools,
    read_resource,
    show_resource_and_prompt_messages,
)
from lib.retry import call_with_retry

default_models = load_environment()
generation_model = default_models["generation_model"]
print(f"Generation model: {generation_model}")

SYSTEM_PROMPT = (
    "You are a helpful coding assistant for the developers of our company. "
    "Use the available tools when a question depends on internal rules or conventions of the company. "
    "If a question has several parts, search separately for each part. "
    "Base your answer only on the search results and cite the sources."
)

### The MCP server

The server is a separate Python script next to this notebook. It uses `FastMCP` from the official [MCP Python SDK](https://github.com/modelcontextprotocol/python-sdk). Compare it with the last lesson:

* The function `search_knowledge_base` is the same – it only got the line `@mcp.tool()` above it (a *decorator*). That's how FastMCP knows that this function is a tool.
* There is **no hand-written tool declaration** like in the last lesson. FastMCP builds it from the **type hints** (`query: str`) and the **docstring**.
* The search itself (the index of the knowledge base, like on day 1) comes from the shared code in `lib`.
* Logs go to `stderr`, never to the normal output: the normal output (`stdout`) carries the messages of the protocol. A single `print()` there would break the communication.

In [ ]:
SERVER_SCRIPT = PROJECT_ROOT / "day-02" / "solutions" / "02_mcp" / "knowledge_base_mcp_server.py"
print(SERVER_SCRIPT.read_text())

For the client, a server is just a name and the script to start:

In [ ]:
knowledge_base_server = McpServer(name="knowledge-base", script=SERVER_SCRIPT)

### Under the hood: talking to the server by hand
MCP over `stdio` is surprisingly simple: the client starts the server as a subprocess and both exchange **one JSON-RPC message per line** via stdin and stdout. Before using the SDK, let's do it by hand to see what is exchanged.

The conversation always starts with a handshake:
1. `initialize` – the client tells the server which protocol version and capabilities it supports; the server answers with its name and capabilities.
2. `notifications/initialized` – the client confirms. (Notifications have no `id` and get no answer.)

After that, the client can list and call tools.

>**Note:** The server process needs the API key for the embedding model. A subprocess only inherits environment variables we pass to it, so we pass the current environment.
>
>The first tool call builds the index of the knowledge base (with the embedding cache of the repository, this takes a few seconds).

In [ ]:
server_log = open(SERVER_LOG, "w+")
server_process = subprocess.Popen(
    [sys.executable, str(SERVER_SCRIPT)],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=server_log,
    text=True,
    env=dict(os.environ),
)


def send(message: dict) -> None:
    """Sends one JSON-RPC message to the server (one line on stdin)."""
    print(f">>> {json.dumps(message)}")
    server_process.stdin.write(json.dumps(message) + "\n")
    server_process.stdin.flush()


def receive() -> dict:
    """Reads one JSON-RPC message from the server (one line on stdout)."""
    message = json.loads(server_process.stdout.readline())
    print(f"<<< {json.dumps(message)[:800]}\n")
    return message

In [ ]:
# 1. Handshake
send({
    "jsonrpc": "2.0",
    "id": 1,
    "method": "initialize",
    "params": {
        "protocolVersion": LATEST_PROTOCOL_VERSION,
        "capabilities": {},
        "clientInfo": {"name": "diy-copilot-notebook", "version": "1.0"},
    },
})
receive()
send({"jsonrpc": "2.0", "method": "notifications/initialized"})

In [ ]:
# TODO: 2. Ask the server which tools it offers (method "tools/list")
send({"jsonrpc": "2.0", "id": 2, "method": "tools/list"})
tools_list = receive()
print(json.dumps(tools_list["result"]["tools"][0], indent=1))

In [ ]:
# TODO: 3. Call the tool (method "tools/call" with the tool name and the arguments)
send({
    "jsonrpc": "2.0",
    "id": 3,
    "method": "tools/call",
    "params": {"name": "search_knowledge_base", "arguments": {"query": "maximum duration of a meeting room booking in Berlin"}},
})
tool_result = receive()

In [ ]:
# Stop the server and have a look at its log (stderr)
server_process.terminate()
server_process.wait()
server_log.seek(0)
print(server_log.read()[-1500:])

Compare the result of `tools/list` with the tool declaration you wrote by hand in the last lesson: it's the same information – name, description and a JSON schema of the parameters (`inputSchema`). The difference: the **server** provides it, and the client **discovers** it at runtime.

### Exercise 01: Discover and call the tools of a server

Of course nobody writes these messages by hand. The MCP SDK does it for us; we wrapped it in two small functions:

* `await list_tools(server)` – asks the server which tools it offers
* `await call_tool(server, tool_name, arguments)` – calls a tool and returns its result as text

>**What's `await`?** Talking to another process takes time. Functions that wait for something are often *asynchronous* in Python, and you call them with `await` in front. In a notebook you can simply write `result = await some_function(...)` – think of it as "wait for the result".

Your task is to list the tools of the knowledge base server and print them with `print_tools`.

In [ ]:
# TODO: List the tools of the knowledge base server and print them
tools = await list_tools(knowledge_base_server)
print_tools(tools)

Now call the tool `search_knowledge_base` via MCP with a question of your choice. The arguments are a dictionary: parameter name → value.

In [ ]:
# TODO: Call the tool via MCP and print the result
result = await call_tool(knowledge_base_server, "search_knowledge_base", {"query": "Which header do internal APIs use for API keys?"})
print(result[:1500])

### Example: MCP tools in the assistant

Now we connect the assistant to the server. `run_mcp_assistant` is the tool loop from the last lesson – read it and compare. There are only two real differences:

1. The tools are **discovered** with `list_tools` instead of written in our code. `mcp_tool_to_openai` converts each MCP tool into the format of the tool declarations from the last lesson – it's the same information.
2. The **server executes** the function (`call_tool`) – not our harness.

The function already works with several servers: it remembers which server offers which tool (`server_for_tool`), so that it can send every tool call to the right server. We'll need that in exercise 04.

In [ ]:
def mcp_tool_to_openai(tool) -> dict:
    """Converts a tool of an MCP server into a tool declaration for the model (OpenAI format, like in the last lesson)."""
    return {
        "type": "function",
        "function": {"name": tool.name, "description": tool.description, "parameters": tool.inputSchema},
    }


async def run_mcp_assistant(question: str, servers: list[McpServer], max_steps: int = 6) -> str:
    """Answers a question using the tools of one or more MCP servers."""
    # Difference 1: discover the tools of all servers - and remember which server offers which tool
    tool_declarations = []
    server_for_tool = {}
    for server in servers:
        for tool in await list_tools(server):
            tool_declarations.append(mcp_tool_to_openai(tool))
            server_for_tool[tool.name] = server

    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        response = call_with_retry(completion, model=generation_model, messages=messages, tools=tool_declarations, temperature=0.0)
        message = response.choices[0].message
        messages.append(message.model_dump())

        if not message.tool_calls:
            return message.content

        for tool_call in message.tool_calls:
            name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            if name not in server_for_tool:
                result = f"Error: unknown tool '{name}'."
            else:
                server = server_for_tool[name]
                print(f"[step {step}] {name}({arguments})  -> MCP server '{server.name}'")
                # Difference 2: the MCP server executes the function
                result = await call_tool(server, name, arguments)
            messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": result})

    return "I could not finish the task within the allowed number of steps."

In [ ]:
answer = await run_mcp_assistant(
    "A developer wants to cancel a meeting room booking in the Berlin office 1 hour before it starts via our API. "
    "Is that allowed, and which HTTP status code should the API return if it is not?",
    servers=[knowledge_base_server],
)
print(f"\n{answer}")

Our assistant now uses a tool it doesn't contain. If the team that owns the knowledge base adds a new tool to the server (e.g. `get_document(path)`), our assistant can use it **without any change** – it discovers it the next time it asks the server.

### Exercise 02: Write your own MCP server

Now it's your turn: connect your own data. Team Workplace has an export of its issue tracker (you know the tickets from the prompting lesson on day 1). Let's make it available to every assistant with a small MCP server offering two tools:

* `search_tickets(query, status)` – finds tickets whose title or description contains the query, optionally only tickets with a given status (already done)
* `get_ticket(ticket_id)` – returns all details of one ticket

Your tasks:
1. Implement `get_ticket`: find the ticket with the given id in the list `TICKETS` and return it; if there is none, return an error message.
2. Write good **docstrings** for both tools. Remember: the docstring becomes the tool description – it's the only thing the model knows about the tool.

We write the server directly from the notebook into a file. If the first line of a code cell is `%%writefile <path>`, Jupyter doesn't run the cell as Python – it saves the rest of the cell as a file under this path (`{TICKETS_SERVER_SCRIPT}` is replaced by the value of the variable). The path of the ticket export is passed to the server as a command line argument (`sys.argv[1]`).

In [ ]:
TICKETS_SERVER_SCRIPT = Path(tempfile.gettempdir()) / "tickets_mcp_server.py"
print(TICKETS_SERVER_SCRIPT)

In [ ]:
%%writefile {TICKETS_SERVER_SCRIPT}
"""MCP server for the issue tracker export of Team Workplace."""

import json
import sys
from pathlib import Path

from mcp.server.fastmcp import FastMCP

# The list of all tickets - each ticket is a dictionary with id, title, status, description, ...
TICKETS = json.loads(Path(sys.argv[1]).read_text())

mcp = FastMCP("team-workplace-tickets", log_level="WARNING")


# TODO: Write a good docstring
@mcp.tool()
def search_tickets(query: str = "", status: str = "") -> str:
    """Searches the tickets of Team Workplace (the team that owns the Roomie service).

    Args:
        query: Text to search for in the title and the description, e.g. an error code or a component.
        status: Optional status filter: "open", "in progress", "done" or "rejected". Empty: all tickets.

    Returns:
        A list of matching tickets with id, status and title.
    """
    results = []
    for ticket in TICKETS:
        text = (ticket["title"] + " " + ticket["description"]).lower()
        # every word of the query must occur in the title or the description
        found = all(word in text for word in query.lower().split())
        if found and (status == "" or ticket["status"] == status):
            results.append(f"{ticket['id']} [{ticket['status']}] {ticket['title']}")
    return "\n".join(results) or "No tickets found."


# TODO: Implement get_ticket and write a good docstring
@mcp.tool()
def get_ticket(ticket_id: str) -> str:
    """Returns all details of a ticket: description, status, workaround and resolution.

    Args:
        ticket_id: The id of the ticket, e.g. "WORK-412".
    """
    for ticket in TICKETS:
        if ticket["id"] == ticket_id:   # TODO: Update condition
            return json.dumps(ticket, indent=1)
    return f"Error: the ticket {ticket_id} does not exist."


if __name__ == "__main__":
    mcp.run()

Let's test the new server: which tools does it offer – and what did FastMCP make of your docstrings?

In [ ]:
tickets_server = McpServer(name="tickets", script=TICKETS_SERVER_SCRIPT, args=[str(TICKETS_PATH)])

print_tools(await list_tools(tickets_server))
print(await call_tool(tickets_server, "search_tickets", {"query": "500", "status": "open"}))
print(await call_tool(tickets_server, "get_ticket", {"ticket_id": "WORK-412"}))

### (Optional) Exercise 03: Resources and prompts

Tools are only one of three things an MCP server can offer. The difference is **who decides** when they are used:

| | Who decides? | Example in Claude Code | Our ticket server |
|---|---|---|---|
| **Tools** | the **model** calls them when it thinks it needs them | the model searches the tickets | `search_tickets`, `get_ticket` |
| **Resources** | the **user or the harness** chooses data that goes into the context | mention a resource with `@tickets:ticket://WORK-412` | every ticket as `ticket://<id>` |
| **Prompts** | the **user** starts a prepared prompt template | slash command `/mcp__tickets__triage_ticket WORK-421` | `triage_ticket(ticket_id)` |

Resources are context that the *user* controls – no tool call, no guessing by the model. Prompts are reusable instructions of a team, offered by the server – everybody who connects the server gets them.

With FastMCP, resources and prompts are functions with a decorator, just like tools:
* `@mcp.resource("ticket://{ticket_id}")` – the part in curly braces becomes the parameter of the function
* `@mcp.prompt()` – the function returns the text of the prompt

We don't rewrite the server of exercise 02. The new file **imports** it and adds a resource and a prompt to it.

Your task is to write the **prompt template** `triage_ticket`: it gets a ticket id and returns a prompt that asks the model to triage the ticket – with the ticket details and a clear answer format.

In [ ]:
TICKETS_SERVER_V2_SCRIPT = Path(tempfile.gettempdir()) / "tickets_mcp_server_v2.py"

In [ ]:
%%writefile {TICKETS_SERVER_V2_SCRIPT}
"""The ticket server of exercise 02 - extended by a resource and a prompt."""

from tickets_mcp_server import get_ticket, mcp


@mcp.resource("ticket://{ticket_id}")
def ticket(ticket_id: str) -> str:
    """A ticket of Team Workplace with all details."""
    return get_ticket(ticket_id)


# TODO: Write the prompt template
@mcp.prompt()
def triage_ticket(ticket_id: str) -> str:
    """Triage a bug ticket of Team Workplace: severity, affected users, workaround and next steps."""
    return f"""You are triaging a bug ticket of Team Workplace, the team that owns Roomie, the meeting room booking service.

The ticket:
{get_ticket(ticket_id)}

Answer in exactly this format:
Severity: S1, S2, S3 or S4 - and one sentence why
  (S1: nobody can book or data of other users is exposed, S2: a core function fails without workaround,
   S3: a function fails but there is a workaround, S4: cosmetic or rare)
Affected users: who notices the bug
Workaround: what users can do until it is fixed, or "none"
Next steps: at most three bullet points"""


if __name__ == "__main__":
    mcp.run()

What does the server offer now? And what do the messages for reading a resource and getting a prompt look like?

In [ ]:
tickets_server_v2 = McpServer(name="tickets", script=TICKETS_SERVER_V2_SCRIPT, args=[str(TICKETS_PATH)])

print_resources(await list_resources(tickets_server_v2))
print_prompts(await list_prompts(tickets_server_v2))
show_resource_and_prompt_messages(tickets_server_v2, "ticket://WORK-412", "triage_ticket", {"ticket_id": "WORK-421"})

**Using a resource:** the developer mentions a ticket in the message, e.g. `@WORK-412`. The harness reads the resource and attaches it to the message – the model gets the ticket right away, without calling a tool.

Your task is to attach the resource to the message of the developer.

In [ ]:
question = "@WORK-412 What can the mobile team do until this bug is fixed? Two sentences."

# TODO: Read the resource and attach it to the message
ticket_text = await read_resource(tickets_server_v2, "ticket://WORK-412")
user_message = f"{question}\n\nAttached resource ticket://WORK-412:\n{ticket_text}"

response = call_with_retry(
    completion,
    model=generation_model,
    messages=[{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_message}],
    temperature=0.0,
)
print(response.choices[0].message.content)

**Using a prompt:** the developer starts the prompt `triage_ticket` for a ticket (in Claude Code: a slash command). The server returns ready-made messages – the harness sends them to the model as they are.

In [ ]:
messages = await get_prompt(tickets_server_v2, "triage_ticket", {"ticket_id": "WORK-421"})
print(f"Messages from the server:\n{messages[0]['content'][:600]} ...\n")

response = call_with_retry(completion, model=generation_model, messages=messages, temperature=0.0)
print(response.choices[0].message.content)

What did we learn?
* An MCP server can offer **tools** (the model decides), **resources** (the user or harness decides) and **prompts** (the user decides).
* Resources put exactly the context into the request that the user chose – cheap and predictable, no tool call needed.
* Prompts share the instructions of a team: everybody who connects the server gets the same triage prompt. In the context engineering lesson you will meet a similar idea, *skills* – the difference: a prompt is started by the user, a skill is loaded by the model when it needs it.

### Additional information: Several servers at once

Real harnesses like Claude Code are connected to many MCP servers at the same time, and the model sees the tools of all of them. The harness remembers which tool belongs to which server, so that it can send each call to the right one – our `run_mcp_assistant` does that with the dictionary `server_for_tool`: just pass a list of servers.

The question below can only be answered with **both** servers: a known problem from the tickets *and* a rule from the knowledge base. Watch the output: which server answers which tool call?

>**Note:** Tool names must be unique across all servers. Harnesses often prefix them with the server name (e.g. `tickets__search_tickets`) to avoid collisions.

In [ ]:
# A question that needs both servers: the tickets and the knowledge base
answer = await run_mcp_assistant(
    "Developers report a 500 error when they create a booking from the mobile app. "
    "Is this a known problem, and which status code should the API return instead according to our guidelines?",
    servers=[knowledge_base_server, tickets_server],
)
print(f"\n{answer}")

### Outlook: Use the same server in other harnesses
The real value of MCP: **one server, many clients**. The same `knowledge_base_mcp_server.py` can be added to the AI tools your developers already use. Try one of them on your own machine (not in Colab), in a copy of this repository:

**MCP Inspector** – the "Postman for MCP servers". It lists the tools and lets you call them by hand:
```bash
npx @modelcontextprotocol/inspector python day-02/solutions/02_mcp/knowledge_base_mcp_server.py
```

**Claude Code**:
```bash
claude mcp add company-knowledge-base -- python /path/to/knowledge_base_mcp_server.py
```

**VS Code (GitHub Copilot agent mode)** – in `.vscode/mcp.json`:
```json
{
  "inputs": [
    {"type": "promptString", "id": "gemini-api-key", "description": "Gemini API key", "password": true}
  ],
  "servers": {
    "company-knowledge-base": {
      "type": "stdio",
      "command": "python",
      "args": ["/path/to/knowledge_base_mcp_server.py"],
      "env": {"GEMINI_API_KEY": "${input:gemini-api-key}"}
    }
  }
}
```

Many companies offer hosted MCP servers for their products (e.g. Atlassian for Jira and Confluence, GitHub). They are connected via HTTP instead of `stdio` – for the harness, nothing else changes.

### Security considerations
MCP makes it easy to connect tools – and just as easy to connect the wrong ones. Remember the guardrails lesson:
* **A local MCP server is a program running on your machine** with your permissions. Only install servers you trust.
* **Tool descriptions and tool results go straight into the prompt.** A malicious server can hide instructions in them ("tool poisoning") – the same problem as indirect prompt injection.
* **Remote servers see your data.** Check what is sent to them, especially personal data and secrets (AI-02).
* Tools with side effects need the same checks as before: validation, output guards and human approval.

### Learnings
* MCP standardizes how harnesses **discover** and **call** tools: a server describes its tools, any client can use them.
* Under the hood, MCP is JSON-RPC: `initialize`, `tools/list`, `tools/call` – via stdin/stdout for local servers or HTTP for remote servers.
* FastMCP builds the tool description from the type hints and the docstring – no hand-written JSON. Docstrings are prompts.
* The tool loop doesn't change: the tools come from the server (`list_tools`), and the server executes them (`call_tool`).
* Besides tools, servers offer **resources** (context chosen by the user) and **prompts** (templates started by the user).
* Writing your own MCP server takes only a few lines – that's how you connect your own data and services to any assistant. With several servers, the harness sends each tool call to the right server.
* In a local server, `stdout` belongs to the protocol – log to `stderr`.
* Treat MCP servers like any other dependency: trust, review, and guard what goes in and out.